Load in modules

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import defdap.hrdic as hrdic
import defdap.ebsd as ebsd
import defdap.experiment as experiment

from pathlib import Path

%matplotlib inline

Change a default setting within defdap

In [ ]:
from defdap import defaults
defaults['hrdic_grain_finding_method'] = 'warp'

Define an empty experiment to load data into

In [ ]:
exp = experiment.Experiment()

Define paths to data. HRDIC path now has a wildcard '*' to use all matched files.

In [ ]:
data_dir = Path('./')
hrdic_dir = data_dir / 'DIC_DaVis_Export'
hrdic_file_pattern = 'B*.txt'
ebsd_file = data_dir / 'EBSD/Sample 2.cpr'

Load HRDIC maps and place them in a new increment (default) in the experiment, but define a single frame they share to make registration easier.

In [ ]:
dic_frame = experiment.Frame()
for dic_file in sorted(hrdic_dir.glob(hrdic_file_pattern)):
    hrdic.Map(dic_file, experiment=exp, frame=dic_frame)

Load the EBSD and place it in the first incremnet but in a different frame.

In [ ]:
ebsd_frame = experiment.Frame()
ebsd.Map(
    ebsd_file,
    increment=exp.increments[0], frame=experiment.Frame()
)

Define homologous points. This only needs to be done for once for all of the HRDIC maps as they share a reference frame.

In [ ]:
dic_map = exp.increments[0].maps['hrdic']
dic_map.frame.homog_points = [
    ( 806,  239), (1198,  147), (1445,  376), (1769,  790), (1622, 1341),
    (1782, 1814), (1215, 1473), ( 563, 1617), ( 728, 1295), ( 758, 1102),
    ( 483, 1161), ( 231, 1032), ( 212,  659), (1169, 1066)
]

ebsd_map = exp.increments[0].maps['ebsd']
ebsd_map.frame.homog_points = [
    (238, 120), (324, 102), (375, 147), (440, 230), (400, 344), 
    (429, 440), (310, 371), (166, 399), (207, 334), (216, 295), 
    (155, 308), (102, 282), (104, 206), (306, 288)
]

Loop over all the HRIDC maps stored in the experiment to set some parameters.

In [ ]:
field_width = 30 # microns
num_pixels = 2048
pixel_size = field_width / num_pixels

for inc, dic_map in exp.iter_over_maps('hrdic'):
    dic_map.set_crop(left=100, right=100, bottom=100, top=100)
    dic_map.link_ebsd_map(ebsd_map, transform_type="affine")
    dic_map.set_scale(pixel_size)

Loop over the maps and plot a subregion of each map.

In [ ]:
for inc, dic_map in exp.iter_over_maps('hrdic'):
    plot = dic_map.plot_map(
        'max_shear', vmin=0, vmax=0.10, 
        plot_scale_bar=False, plot_gbs='line'
    )
    plot.ax.set_xlim([800, 1200])
    plot.ax.set_ylim([800, 1200])
    plt.tight_layout()

As the grain segmentation is done using the same EBSD data, the grain ids for all of the HRDIC should be consistent. So pick the same grain from each time step and plot deformation data over each step.

In [ ]:
dic_grain_id = 400

num_incs = len(exp.increments)
fig, axes = plt.subplots(2, int(np.ceil(num_incs / 2)))

for inc, dic_map in exp.iter_over_maps('hrdic'):
    dic_grain = dic_map[dic_grain_id]
    plot = dic_grain.plot_map(
        'max_shear', vmin=0, vmax=0.10, 
        plot_scale_bar=(inc == 0), plot_colour_bar=False,
        fig=fig, ax=axes.flat[inc]
    )

for ax in axes.flat:
    ax.axis('off')

fig.subplots_adjust(right=0.8)
cbar_ax = fig.add_axes([0.85, 0.15, 0.03, 0.7])
fig.colorbar(plot.img_layers[0], cax=cbar_ax, label="Effective shear strain")
